In [6]:
import os
from matplotlib import pyplot as plt
import cv2
import numpy as np
from pathlib import Path
import numpy as np
import struct
from flow_vis import flow_to_color
import time

def import_frames(folder_path):
    image_list = []
   
    # Ensure the folder path is valid
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return image_list
   
    # Get a sorted list of filenames in the folder
    sorted_filenames = sorted(os.listdir(folder_path))
   
    # Iterate through sorted files in the folder
    for filename in sorted_filenames:
        file_path = os.path.join(folder_path, filename)
       
        # Check if the file is an image (you can add more image extensions if needed)
        if file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            # Read the image
     
            img = cv2.imread(file_path)
           
            # Append the image to the list
            if img is not None:
                #gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                image_list.append(img)
            else:
                print(f"Error reading image: {filename}")
   
    return image_list



def create_video(images_lists, text_list, output_path, fps=24):
    # Get the height and width of the frames
    height, width = images_lists[0][0].shape[:2]

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (2 * width, 2 * height), isColor=True)

    # Iterate through the frames in each list
    for i in range(len(images_lists[0])):
        # Create a 2 by 2 grid by concatenating images horizontally and vertically
        top_row = np.concatenate((images_lists[0][i], images_lists[1][i]), axis=1)
        bottom_row = np.concatenate((images_lists[2][i], images_lists[3][i]), axis=1)
        final_frame = np.concatenate((top_row, bottom_row), axis=0)
       
       
        # Add text to each space in the grid
        for j, text in enumerate(text_list):
            text_position = (width * (j % 2), height * (j // 2) + 20)
            cv2.putText(final_frame, str(text), text_position, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
 
        # Write the frame to the video file
        video_writer.write(final_frame)

    # Release the VideoWriter object
    video_writer.release()


def read_flo(filename):  
    with open(filename, 'rb') as f:
        magic, = np.fromfile(f, np.float32, count=1)
        
        if 202021.25 != magic:
            print('Magic number incorrect. Invalid .flo file')
        else:
            w = np.fromfile(f, np.int32, count=1)[0]
            h = np.fromfile(f, np.int32, count=1)[0]

            # Read flow data
            data = np.fromfile(f, np.float32, count=2*w*h)
            # Reshape to get the flow field
            data2D = np.resize(data, (h, w, 2))
            return data2D

        
    
def extract_metrics(ground_truth_flows, estimated_flows):
    MRSEs = []
    MAEs = []
    angular_errors = []
    EPEs = []
    for ground_truth_flow, estimated_flow in zip(ground_truth_flows, estimated_flows):
        MRSE = np.sqrt(np.mean(np.square(np.linalg.norm(ground_truth_flow - estimated_flow, axis=-1))))
        MAE = np.mean(np.linalg.norm(ground_truth_flow - estimated_flow, axis=-1))
        EPE = np.linalg.norm(ground_truth_flow - estimated_flow, axis=-1)

        MRSEs.append(MRSE)
        MAEs.append(MAE)
        EPEs.append(EPE)
    
    print(f"MRSE: {np.mean(MRSEs)}")
    print(f"MAE: {np.mean(MAEs)}")
    print(f"EPE: {np.mean(EPEs)}")

In [2]:
# Reading frames
path_fr = 'D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 3/training/temple_2/'
frames_list = import_frames(path_fr)

# Reading ground truth optical-flow png
path_gt_viz = 'D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 3/flow_viz/temple_2/'
gt_viz_list = import_frames(path_gt_viz)

# Reading ground truth optical-flow
path_gt = 'D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 3/flow/temple_2/'
gt_list = os.listdir(path_gt)

gt_frames = []
for gt in gt_list:
    
    frame = read_flo(path_gt + gt)
    gt_frames.append(frame)

In [3]:
# Optical flow

estimated_flow_frames = []
estimated_flow_v2_frames = []
estimated_flow_v3_frames = []

estimated_color_flow_frames = []
estimated_color_flow_v2_frames = []
estimated_color_flow_v3_frames = []

for i in range(1, len(frames_list)):
    
    prev_fr = frames_list[i-1]
    next_fr = frames_list[i]
    
    # Calculate Optical Flow
    start_diff = time.time()
    flow = cv2.optflow.calcOpticalFlowDenseRLOF(prev_fr, next_fr, None)
    time_diff = round(time.time() - start_diff, 3)
    estimated_flow_frames.append(flow)
    
    # Calculate Optical Flow v2
    start_diff = time.time()
    flow_v2 = cv2.optflow.calcOpticalFlowDenseRLOF(prev_fr, next_fr, None, gridStep=(4,4))
    time_diff_v2 = round(time.time() - start_diff, 3)
    estimated_flow_v2_frames.append(flow_v2)
    
    # Calculate Optical Flow v3
    start_diff = time.time()
    flow_v3 = cv2.optflow.calcOpticalFlowDenseRLOF(prev_fr, next_fr, None, gridStep=(12,12)) 
    time_diff_v3 = round(time.time() - start_diff, 3)
    estimated_flow_v3_frames.append(flow_v3)
    
    # Convert to HSV
    flow_color = flow_to_color(flow)
    flow_color_v2 = flow_to_color(flow_v2)
    flow_color_v3 = flow_to_color(flow_v3)
    
    estimated_color_flow_frames.append(flow_color)
    estimated_color_flow_v2_frames.append(flow_color_v2)
    estimated_color_flow_v3_frames.append(flow_color_v3)
    

In [4]:
# Extracting metrics
print("Optical flow RLOF")
print(f"Computational time: {time_diff}")
extract_metrics(gt_frames, estimated_flow_frames)

print("\nOptical flow RLOF v2")
print(f"Computational time: {time_diff_v2}")
extract_metrics(gt_frames, estimated_flow_v2_frames)

print("\nOptical flow RLOF v3")
print(f"Computational time: {time_diff_v3}")
extract_metrics(gt_frames, estimated_flow_v3_frames)

Optical flow RLOF
Computational time: 0.432
MRSE: 11.29072380065918
MAE: 4.676564693450928
EPE: 4.676568031311035

Optical flow RLOF v2
Computational time: 0.618
MRSE: 10.964408874511719
MAE: 4.348351001739502
EPE: 4.3483452796936035

Optical flow RLOF v3
Computational time: 0.136
MRSE: 12.472161293029785
MAE: 5.769882678985596
EPE: 5.769876956939697


In [5]:
# Saving video
output_video_path = 'D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 3/output_opt_flow_RLOF.mp4'
sequence = [gt_viz_list, estimated_color_flow_frames, 
            estimated_color_flow_v2_frames, estimated_color_flow_v3_frames]

create_video(sequence, ["Ground truth", "OpticalFlowDenseRLOF",
                        "OpticalFlowDenseRLOF_v2","OpticalFlowDenseRLOF_v3"], output_video_path, fps=12)
